In [30]:
!pip install -q torch-geometric scikit-learn pandas numpy tqdm

import pandas as pd
import numpy as np
import glob
from tqdm import tqdm
import torch
import torch.nn as nn
import random

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv, RGCNConv

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [31]:
MAX_ROWS = 100000

data_list = []
count = 0

# KT1
for i, file in enumerate(glob.glob("/kaggle/input/datasets/jasperamarlapudi/interactive-kt1-ednet/KT1/u*.csv")):
    df = pd.read_csv(file).head(200)
    df['user_id'] = i
    df['correct'] = (df['user_answer'] != -1).astype(int)
    
    data_list.append(df[['user_id','question_id','timestamp','correct']])
    count += len(df)
    
    if count > MAX_ROWS//2:
        break

# KT3
for i, file in enumerate(glob.glob("/kaggle/input/datasets/youssefelbadouri/ednet-kt3-micro-behaviors-dataset/KT3/*.csv")):
    df = pd.read_csv(file)
    df = df[df['item_id'].str.startswith('q')]
    df = df[df['action_type'] == 'respond']
    df = df.head(200)
    
    df['question_id'] = df['item_id']
    df['user_id'] = i + 100000
    df['correct'] = df['user_answer'].notnull().astype(int)
    
    data_list.append(df[['user_id','question_id','timestamp','correct']])
    count += len(df)
    
    if count > MAX_ROWS:
        break

data = pd.concat(data_list).reset_index(drop=True)
print("Total rows:", len(data))

Total rows: 100090


In [32]:
data['skill'] = data['question_id'].astype('category').cat.codes % 50

In [33]:
user_map = {u:i for i,u in enumerate(data['user_id'].unique())}
ques_map = {q:i+len(user_map) for i,q in enumerate(data['question_id'].unique())}
skill_map = {s:i+len(user_map)+len(ques_map) for i,s in enumerate(data['skill'].unique())}

edges = []
labels = []

for row in data.itertuples():
    u = user_map[row.user_id]
    q = ques_map[row.question_id]
    s = skill_map[row.skill]
    
    edges.append([u,q])
    edges.append([q,s])
    labels.append(row.correct)

edge_index = torch.tensor(edges, dtype=torch.long).t()

y = torch.tensor(labels, dtype=torch.float)
y = y.repeat(2)[:edge_index.shape[1]]

# limit edges for speed
MAX_EDGES = 200000
edge_index = edge_index[:, :MAX_EDGES]
y = y[:MAX_EDGES]

x = torch.randn((len(user_map)+len(ques_map)+len(skill_map), 32))

data_graph = Data(x=x, edge_index=edge_index, y=y).to(device)

In [34]:
indices = np.arange(edge_index.shape[1])

train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_edge_index = edge_index[:, train_idx]
train_y = y[train_idx]

test_edge_index = edge_index[:, test_idx]
test_y = y[test_idx]

In [35]:
num_nodes = x.shape[0]

def generate_negative_edges(num_samples, existing_edges):
    neg_edges = []
    existing = set([(int(u), int(v)) for u,v in zip(existing_edges[0], existing_edges[1])])

    while len(neg_edges) < num_samples:
        u = random.randint(0, num_nodes-1)
        v = random.randint(0, num_nodes-1)
        
        if (u,v) not in existing:
            neg_edges.append([u,v])

    return torch.tensor(neg_edges).t()

# train
train_neg = generate_negative_edges(train_edge_index.shape[1], train_edge_index)

train_edge_index_full = torch.cat([train_edge_index, train_neg], dim=1)

train_y_full = torch.cat([
    torch.ones(train_edge_index.shape[1]),
    torch.zeros(train_neg.shape[1])
]).to(device)

# test
test_neg = generate_negative_edges(test_edge_index.shape[1], test_edge_index)
test_edge_index_full = torch.cat([test_edge_index, test_neg], dim=1)

test_y_full = torch.cat([
    torch.ones(test_edge_index.shape[1]),
    torch.zeros(test_neg.shape[1])
])

In [36]:
class GCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = GCNConv(32,32)
    def forward(self,x,e):
        return torch.relu(self.conv(x,e))

class GAT(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = GATConv(32,32)
    def forward(self,x,e):
        return torch.relu(self.conv(x,e))

class SAGE(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = SAGEConv(32,32)
    def forward(self,x,e):
        return torch.relu(self.conv(x,e))

class GIN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = GINConv(nn.Sequential(nn.Linear(32,32)))
    def forward(self,x,e):
        return torch.relu(self.conv(x,e))

class RGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = RGCNConv(32,32,2)
    def forward(self,x,e):
        edge_type = torch.zeros(e.size(1),dtype=torch.long).to(device)
        return torch.relu(self.conv(x,e,edge_type))

class TGAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(32,32,batch_first=True)
    def forward(self,x):
        x,_ = self.lstm(x.unsqueeze(0))
        return x.squeeze(0)

In [40]:
def train(model):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.BCEWithLogitsLoss()

    src = train_edge_index_full[0]
    dst = train_edge_index_full[1]

    for epoch in range(40):
        opt.zero_grad()

        if isinstance(model, TGAN):
            out = model(data_graph.x)
        else:
            out = model(data_graph.x, data_graph.edge_index)

        edge_feat = torch.cat([out[src], out[dst]], dim=1)

        edge_pred = nn.Linear(edge_feat.shape[1], 1).to(device)(edge_feat).squeeze()

        loss = loss_fn(edge_pred, train_y_full)

        loss.backward()
        opt.step()

    return model

In [41]:
pos_weight = torch.tensor([2.0]).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [42]:
models = [GCN(), GAT(), SAGE(), GIN(), RGCN()]
outputs = []

for m in models:
    m = train(m)
    out = m(data_graph.x, data_graph.edge_index)
    src = test_edge_index_full[0]
    dst = test_edge_index_full[1]
    outputs.append(torch.sigmoid((out[src]*out[dst]).sum(dim=1)).detach())

tgan = train(TGAN())
out = tgan(data_graph.x)
outputs.append(torch.sigmoid((out[src]*out[dst]).sum(dim=1)).detach())

In [44]:
final = sum(outputs)/len(outputs)

pred = (final > 0.4).int().cpu()
true = test_y_full.cpu().int()

print("Accuracy:", accuracy_score(true,pred))
print("Precision:", precision_score(true,pred))
print("Recall:", recall_score(true,pred))
print("F1:", f1_score(true,pred))

Accuracy: 0.5
Precision: 0.5
Recall: 1.0
F1: 0.6666666666666666
